In [ ]:
from google.colab import files
uploaded = files.upload()

Saving braille detection.tensorflowresnet.zip to braille detection.tensorflowresnet.zip


In [ ]:
import os
for name, data in uploaded.items():
    print(name, len(data), "bytes")

braille detection.tensorflowresnet.zip 484904900 bytes


In [ ]:
import zipfile
import os

zip_path = "/content/braille detection.tensorflowresnet.zip"  # adjust filename to match what you uploaded
# or if using Drive: zip_path = "/content/drive/MyDrive/your-project.v1-tfrecord.zip"

extract_dir = "roboflow_dataset"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(os.listdir(extract_dir))

['README.dataset.txt', 'test', 'README.roboflow.txt', 'valid', 'train']


In [ ]:
print(os.listdir(f"{extract_dir}/train"))
print(os.listdir(f"{extract_dir}/valid"))


['IMG20230503105457_jpg.rf.TVriGrJaFlEWI2yzgsIx.jpg', 'db33_png.rf.QIxBXP2NoTCWD2a2yOHA.png', '902_jpg.rf.iz059t36BqQHwx6yJngP.jpg', '610_jpg.rf.hKo2YrBDjxveAjF66Xe0.jpg', '0000007_jpg.rf.tNts8hGXTj8YtG71w4Bv.jpg', 'IMG20230422143548_jpg.rf.dinNwqv013WubFXZ7TeM.jpg', '4580_jpg.rf.Rrjc2MFcC1ZzBAluFONU.jpg', '3856_jpg.rf.UuUNcBGebMRWtNslLJGS.jpg', '2698_jpg.rf.VJ8qRN7QC31B8UqwPOIA.jpg', 'IMG20230422143541_jpg.rf.Dsj8sDFyiLbBc4RG8xdX.jpg', '2877_jpg.rf.cU47NZW0k1KFZG1UmqjN.jpg', '537_jpg.rf.kV7OV5ntqWyFtrSClXB3.jpg', '3293_jpg.rf.SmsgwqasxXGVyPXJFHm0.jpg', '1561_jpg.rf.BdySqqYfdEeVjbfhcoSG.jpg', '1365_jpg.rf.53y275Ckw8Pg4IsMg6bX.jpg', '360_F_184511922_9ox3DVyRV1F5cvYzunAhYEgoVsjt9RB9_jpg.rf.h1JMH2iwKoCXzsFd3men.jpg', '618_jpg.rf.2rOBxnsawxCzelYJBQSS.jpg', '3282_jpg.rf.iJwgoBi1nBgv0MoSmN5F.jpg', '3503_jpg.rf.JiQeRXQjV26jCfkaAhEh.jpg', '4059_jpg.rf.2vwl7fPx9MyJlJsE8Yic.jpg', '1566_jpg.rf.RAE3Fkq83AWH0eiHsjCy.jpg', '4566_jpg.rf.Rv5mBZik73FgU6pGGYPp.jpg', 'IMG_6244_jpg.rf.26ECWjR6ABEffZAVbvMt

In [ ]:
import tensorflow as tf
import os
from PIL import Image
import io

def parse_label_map(pbtxt_path):
    id_to_name = {}
    with open(pbtxt_path) as f:
        content = f.read()
    items = content.split("item {")[1:]
    for item in items:
        id_line = [l for l in item.split("\n") if "id:" in l][0]
        name_line = [l for l in item.split("\n") if "name:" in l][0]
        idx = int(id_line.split(":")[1].strip())
        name = name_line.split(":")[1].strip().strip("'\"")
        id_to_name[idx] = name
    return id_to_name

def crop_tfrecord_to_classification(tfrecord_path, label_map_path, output_dir):
    id_to_name = parse_label_map(label_map_path)
    raw_dataset = tf.data.TFRecordDataset(tfrecord_path)

    feature_desc = {
        'image/encoded': tf.io.FixedLenFeature([], tf.string),
        'image/object/bbox/xmin': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/xmax': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/ymin': tf.io.VarLenFeature(tf.float32),
        'image/object/bbox/ymax': tf.io.VarLenFeature(tf.float32),
        'image/object/class/label': tf.io.VarLenFeature(tf.int64),
    }

    for i, raw_record in enumerate(raw_dataset):
        example = tf.io.parse_single_example(raw_record, feature_desc)
        img = Image.open(io.BytesIO(example['image/encoded'].numpy())).convert("RGB")
        w, h = img.size

        xmins = tf.sparse.to_dense(example['image/object/bbox/xmin']).numpy()
        xmaxs = tf.sparse.to_dense(example['image/object/bbox/xmax']).numpy()
        ymins = tf.sparse.to_dense(example['image/object/bbox/ymin']).numpy()
        ymaxs = tf.sparse.to_dense(example['image/object/bbox/ymax']).numpy()
        labels = tf.sparse.to_dense(example['image/object/class/label']).numpy()

        for j in range(len(labels)):
            box = (xmins[j]*w, ymins[j]*h, xmaxs[j]*w, ymaxs[j]*h)
            cropped = img.crop(box)

            class_name = id_to_name.get(labels[j], f"class_{labels[j]}")
            class_dir = os.path.join(output_dir, class_name)
            os.makedirs(class_dir, exist_ok=True)
            cropped.save(os.path.join(class_dir, f"{i}_{j}.jpg"))

In [ ]:
import os

for root, dirs, files in os.walk("roboflow_dataset"):
    for f in files:
        print(os.path.join(root, f))

roboflow_dataset/README.dataset.txt
roboflow_dataset/README.roboflow.txt
roboflow_dataset/test/IMG20230421112754_jpg.rf.QtpGXnP4PsSUC6NvWVXs.jpg
roboflow_dataset/test/IMG20230421204817_jpg.rf.4zDtjBr8ifCSAVDD9nVh.jpg
roboflow_dataset/test/IMG20230421205225_jpg.rf.Us5BLK0Ijxxv4J2TVLIF.jpg
roboflow_dataset/test/IMG20230421205811_jpg.rf.pJLmhTmZTJJLdSnJfjtC.jpg
roboflow_dataset/test/IMG20230422143915_jpg.rf.Y5tW2AGOiUntbT6c3Ckm.jpg
roboflow_dataset/test/3269_jpg.rf.jVuPGWBXud1tNVW0e5Ox.jpg
roboflow_dataset/test/3481_jpg.rf.QGZaZboLpw8ig8JBtNhs.jpg
roboflow_dataset/test/IMG20230421113719_jpg.rf.2ow6LQ2kr8ZY9yT53T22.jpg
roboflow_dataset/test/IMG20230420223850_jpg.rf.IoFDFbnoDPlPc1V17QgB.jpg
roboflow_dataset/test/850_jpg.rf.OQyAZMRvOCwg8NvzwTfT.jpg
roboflow_dataset/test/IMG20230421113556_jpg.rf.eqVdM60QWqkbTLm287NO.jpg
roboflow_dataset/test/9_jpg.rf.X5LaYcdtliqh1MTmMhYO.jpg
roboflow_dataset/test/db20_png_jpg.rf.gW6Gjxz756KDHz9XBYDN.jpg
roboflow_dataset/test/IMG20230421205235_jpg.rf.ZSyV8yTLW

In [ ]:
import pandas as pd
df = pd.read_csv("roboflow_dataset/train/_annotations.csv")
print(df.columns.tolist())
print(df.head())

['filename', 'width', 'height', 'class', 'xmin', 'ymin', 'xmax', 'ymax']
                                            filename  width  height class  \
0  IMG20230422143607_jpg.rf.smWWmPqvvxJyBOzJJl2A.jpg   4000    1800     R   
1  IMG20230422143607_jpg.rf.smWWmPqvvxJyBOzJJl2A.jpg   4000    1800     O   
2  IMG20230422143607_jpg.rf.smWWmPqvvxJyBOzJJl2A.jpg   4000    1800     U   
3  IMG20230422143607_jpg.rf.smWWmPqvvxJyBOzJJl2A.jpg   4000    1800     N   
4  IMG20230422143607_jpg.rf.smWWmPqvvxJyBOzJJl2A.jpg   4000    1800     D   

   xmin  ymin  xmax  ymax  
0   386    17   528   242  
1   558    18   706   238  
2  1443    25  1592   245  
3  1610    25  1759   247  
4  2145    22  2280   237  


In [ ]:
import pandas as pd
import os
from PIL import Image

def crop_csv_to_classification(split_dir, output_dir):
    csv_path = os.path.join(split_dir, "_annotations.csv")
    df = pd.read_csv(csv_path)

    for idx, row in df.iterrows():
        img_path = os.path.join(split_dir, row["filename"])
        try:
            img = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            print(f"Skipping missing file: {img_path}")
            continue

        box = (row["xmin"], row["ymin"], row["xmax"], row["ymax"])
        cropped = img.crop(box)

        class_name = str(row["class"])
        class_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        crop_filename = f"{idx}_{os.path.basename(row['filename'])}"
        cropped.save(os.path.join(class_dir, crop_filename))

    print(f"Done: {split_dir} -> {output_dir}")

In [ ]:
crop_csv_to_classification("roboflow_dataset/train", "classification_data/train")
crop_csv_to_classification("roboflow_dataset/valid", "classification_data/val")
crop_csv_to_classification("roboflow_dataset/test", "classification_data/test")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Done: roboflow_dataset/train -> classification_data/train
Done: roboflow_dataset/valid -> classification_data/val
Done: roboflow_dataset/test -> classification_data/test


In [ ]:
import os
for cls in os.listdir("classification_data/train"):
    n = len(os.listdir(f"classification_data/train/{cls}"))
    print(cls, n)

N 1536
F 482
M 695
U 668
O 1976
K 648
D 1462
P 604
T 1367
W 362
Y 627
J 253
Q 1285
G 626
S 2511
C 846
Z 257
I 1398
B 642
A 1949
H 1322
E 3170
V 757
X 425
L 1774
R 1376


# RESNET18 Training starts here

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # should print 'cuda' if your Colab GPU is enabled

cpu


In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [ ]:
train_dataset = datasets.ImageFolder("classification_data/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("classification_data/val", transform=val_transforms)
test_dataset = datasets.ImageFolder("classification_data/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)
print(num_classes, class_names)

26 ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 129MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=50):
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")

        for phase, loader in [("train", train_loader), ("val", val_loader)]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == "train":
                scheduler.step()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_corrects.double() / len(loader.dataset)
            print(f"  {phase} loss: {epoch_loss:.4f} acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), "best_resnet18_braille.pt")

    print(f"Best val accuracy: {best_acc:.4f}")
    return model

model = train_model(model, criterion, optimizer, scheduler, num_epochs=20)

Epoch 1/20


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load("best_resnet18_braille.pt"))
model.eval()

correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test accuracy: {correct/total:.4f}")

In [ ]:
torch.save(model.state_dict(), "resnet18_braille_final.pt")
from google.colab import files
files.download("resnet18_braille_final.pt")